In [161]:
!pip install torch_geometric

zsh:1: command not found: pip


In [162]:
!unzip -q /content/fiw_embeddings.zip -d /tmp/
!unzip -q /content/FIDs.zip -d /tmp/

unzip:  cannot find or open /content/fiw_embeddings.zip, /content/fiw_embeddings.zip.zip or /content/fiw_embeddings.zip.ZIP.
unzip:  cannot find or open /content/FIDs.zip, /content/FIDs.zip.zip or /content/FIDs.zip.ZIP.


In [ ]:
import os
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import HeteroData, Data
from torch_geometric.nn import HeteroConv, SAGEConv, GATConv

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import f1_score, classification_report

In [ ]:
ignore_values = {0, -1, 7, 8}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# TODO fix this
# root = Path("/content/drive/MyDrive/ml_with_graphs_project/data")
# familyIDsDir = root / "FIDs"
# embeddingDir = root / "fiw_embeddings"
familyIDsDir = Path("/tmp/FIDs")
embeddingDir = Path("/tmp/fiw_embeddings")

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
import shutil
# local_data_dir = Path("/tmp/fiw_data")

# if not local_data_dir.exists():
#     print("Copying data to local storage...")
#     shutil.copytree(
#         "/content/drive/MyDrive/ml_with_graphs_project/data",
#         local_data_dir
#     )
#     print("Done.")

# embeddingDir  = local_data_dir / "fiw_embeddings"
# familyIDsDir  = local_data_dir / "fiw_families"

local_data_dir = Path("data")
local_embedding_dir = local_data_dir / "fiw_embeddings"
local_families_dir  = local_data_dir / "FIDs"

if not local_embedding_dir.exists():
    family_ids = sorted([p.name for p in Path("/content/drive/MyDrive/ml_with_graphs_project/data/fiw_embeddings").iterdir()])

    for id in tqdm(family_ids, desc="Copying embeddings"):
        for mid_dir in (embeddingDir / id).iterdir():
            src  = mid_dir / "mean_embedding.pkl"
            dst  = local_embedding_dir / id / mid_dir.name / "mean_embedding.pkl"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if src.exists():
                shutil.copy2(src, dst)

if not local_families_dir.exists():
    print("Copying family CSVs...")
    shutil.copytree(familyIDsDir, local_families_dir)
    print("Done.")

# Update paths
embeddingDir = local_embedding_dir
familyIDsDir = local_families_dir


In [ ]:
def load_embedding(path):
  with open(path, "rb") as f:
    emb = pickle.load(f)

  emb = np.asarray(emb)

  if emb.ndim > 1:
    emb = emb.squeeze()

  return emb

def valid_relationship(value):
  if pd.isna(value):
    return False

  try:
    value = int(value)
  except Exception:
    return False

  return value not in ignore_values

def calc_database_relationship(graph_relation, directed):
  if directed:
    return graph_relation + 1
  else:
    # note that 4 would mean EITHER 4 or 1. We can't know exactly which, but we are combining to form a father-child relationship.
    return graph_relation + 2

def calc_graph_relationship(database_relation, directed):
  if directed:
    return database_relation - 1
  else:
    if database_relation == 1:
      database_relation = 4
    elif database_relation == 6:
      database_relation = 3
    elif database_relation == 7 or database_relation == 8:
      database_relation = 6
    return database_relation - 2 # make it start at 0

In [ ]:
from collections import Counter

size_counts = Counter(f['x'].size(0) for f in families)
total_members = sum(size_counts.values())
total_families = sum(size * count for size, count in size_counts.items())
for size, count in sorted(size_counts.items()):
    print(f"  {size} members: {count} families")
print(total_members)
print(total_families)

NameError: name 'families' is not defined

In [ ]:
def find_asymmetric_pairs_from_files():
    asymmetric = set()
    all_relations = set()
    family_ids = sorted([p.name for p in familyIDsDir.iterdir()])

    for id in tqdm(family_ids, desc="Scanning families"):
        mid_csv = familyIDsDir / id / "mid.csv"
        if not mid_csv.exists():
            continue

        try:
            df = pd.read_csv(mid_csv)
            mids = df["MID"].astype(int).tolist()
            row_map = {int(row["MID"]): row for _, row in df.iterrows()}

            for mid_a in mids:
                for mid_b in mids:
                    if mid_a == mid_b:
                        continue
                    if mid_a not in row_map or mid_b not in row_map:
                        continue

                    val_ab = row_map[mid_a].get(str(mid_b))
                    val_ba = row_map[mid_b].get(str(mid_a))

                    if not valid_relationship(val_ab) or not valid_relationship(val_ba):
                        continue

                    rel_ab = int(val_ab)
                    rel_ba = int(val_ba)

                    all_relations.add(rel_ab)
                    all_relations.add(rel_ba)

                    if rel_ab != rel_ba:
                        pair = tuple(sorted([rel_ab, rel_ba]))
                        asymmetric.add(pair)

        except Exception as e:
            print(f"\nFailed on {id}: {e}")

    print("Asymmetric relationship pairs (directed):")
    for a, b in sorted(asymmetric):
        print(f"  {a} <-> {b}")

    print(f"\nAll relation types found: {sorted(all_relations)}")
    print(f"Num relations (raw):      {len(all_relations)}")
    print(f"Num relations (0-indexed):{max(all_relations) + 1}")

    return asymmetric, all_relations

asymmetric_pairs = find_asymmetric_pairs_from_files()
print("Asymmetric relationship pairs (directed):")
for a, b in sorted(asymmetric_pairs):
    print(f"  {a} <-> {b}")


NameError: name 'tqdm' is not defined

# Family Building

In [ ]:
def load_family(id, directed):
  mid_csv = familyIDsDir / id / "mid.csv"
  df = pd.read_csv(mid_csv)

  mids = df["MID"].astype(int).tolist()

  valid_mids = []

  for mid in mids:
    embed_path = embeddingDir / id / f"MID{mid}" / "mean_embedding.pkl"

    if os.path.exists(embed_path):
      valid_mids.append(mid)

  # mid_to_idx = {mid: i for i, mid in enumerate(mids)}
  mid_to_idx = {mid: i for i, mid in enumerate(valid_mids)}

  embeddings = []

  # for mid in mids:
  for mid in valid_mids:
    embed_path = embeddingDir / id / f"MID{mid}" / "mean_embedding.pkl"

    embeddings.append(load_embedding(embed_path))

  x = torch.tensor(np.stack(embeddings), dtype = torch.float)

  edges = []

  # rel_columns = [c for c in df.columns if c not in ["MID", "Name", "Gender"]]

  for _, row in df.iterrows():
    source = int(row["MID"])
    for col in df.columns:
      if col in ["MID", "Name", "Gender"]:
        continue

      fam_member = int(col)

      if source == fam_member:
        continue

      if not valid_relationship(row[col]):
        continue

      fam_rel_id = calc_graph_relationship(int(row[col]), directed)

      if source not in mid_to_idx or fam_member not in mid_to_idx:
        continue

      person = mid_to_idx[source]
      target = mid_to_idx[fam_member]

      edges.append((person, target, fam_rel_id))

  return {
      "family_id": id,
      "x": x,
      "mids": mids,
      "edges": edges,
  }

In [ ]:
from tqdm import tqdm
def load_all_families(directed):
  # family_ids = sorted([p.name for p in familyIDsDir.iterdir()])
  family_ids = sorted([p.name for p in embeddingDir.iterdir()])

  families = []

  for id in tqdm(family_ids, desc = "Loading Families"):
    try:
      mid_csv = familyIDsDir / id / "mid.csv"

      if not mid_csv.exists():
        continue

      family = load_family(id, directed)

      if family['x'].size(0) >= 2 and len(family["edges"]) > 0:
        families.append(family)
    except Exception as e:
      print(f"Error loading family {id}: {e}")

  return families

# Relationship Mapping and Masking

In [ ]:
# as far as I can tell, there are only 0 through 5. 0 is self, so we don't have it.
# instead, I combine 4 and 1 if its undirected (so I replace 1 with 4, and subtract 2 (for 0 and 1) to get the new graph representation)
# if its directed, 1 is just subtracted from the dataset relationship. Planning to test both approaches and compare
# build label mapping is kinda useless I realised, since I can just do the above. BUT, I need to make sure if there are any relationships greater than 5.
# if so, then I might need this since it will work for any number of relationships.

# def build_label_mapping(families):
#   rel_ids = ({rel for family in families for _, _, rel in family["edges"]})
#   rel_to_label = {rel: i for i, rel in enumerate(rel_ids)}
#   label_to_rel = {i, rel for rel, i in rel_to_label.items()}

In [ ]:
def make_masked_sample(family, masked_node, directed):
  x = family['x']

  num_nodes = x.size(0)

  visible_src, visible_dst, visible_rel = [], [], []
  pred_src, pred_dst, pred_rel = [], [], []

  for source, target, relationship in family['edges']:
    if target == masked_node:
      continue

    if source == masked_node:
      pred_src.append(source)
      pred_dst.append(target)
      pred_rel.append(relationship)

    else:
      visible_src.append(source)
      visible_dst.append(target)
      visible_rel.append(relationship)

  if len(pred_rel) == 0:
    return None

  data = Data()
  data.x = x

  data.num_nodes = num_nodes

  if len(visible_src) > 0:
    data.edge_index = torch.tensor([visible_src, visible_dst], dtype = torch.long)
    data.edge_attr = torch.tensor(visible_rel, dtype = torch.long)

  else:
    data.edge_index = torch.empty((2, 0), dtype = torch.long)
    data.edge_attr = torch.empty((0, ), dtype = torch.long)

  data.edge_label_index = torch.tensor([pred_src, pred_dst], dtype = torch.long)
  data.edge_label = torch.tensor(pred_rel, dtype = torch.long)

  data.mask_node = masked_node

  return data

In [ ]:
# # mask_node is the index that we're hiding
# def hetero_make_masked_sample(family, mask_node, rel_to_label, all_rel_ids, directed):
#   # x = family['x']
#   # edges = family['edges']

#   data = HeteroData
#   data['person'].x = family['x']

#   edge_dict = {id: [[], []] for id in all_rel_ids}

#   visible_edges = {}

#   predictions = {
#       "source": [],
#       "target": [],
#       "edge_type": [],
#   }

#   for source, target, relationship in family['edges']:
#     # graph_rel = calc_graph_relationship(relationship, directed)
#     graph_rel = relationship

#     if graph_rel is None:
#       continue

#     if target == mask_node:
#       continue

#     if source == mask_node:
#       predictions['source'].append(source)
#       predictions['target'].append(target)
#       predictions['edge_type'].append(graph_rel)

#       continue

#     visible_edges.setdefault(graph_rel, [[], []])

#     visible_edges[graph_rel][0].append(source)
#     visible_edges[graph_rel][1].append(target)

#   if len(predictions["edge_type"]) == 0:
#     return None

#   all_graph_labels = sorted(set(visible_edges.keys()) | set(predictions["edge_type"]))

#   for relationship in all_graph_labels:
#     edge_type = ("person", f"rel_{relationship}", "person")
#     srcs, tgts = visible_edges.get(relationship, ([], []))

#     if len(srcs) == 0:
#       edge_index = torch.empty((2, 0), dtype = torch.long)

#     else:
#       edge_index = torch.tensor([srcs, tgts], dtype = torch.long)

#     data[edge_type].edge_index = edge_index

#   data["edge_label_index"] = torch.tensor([predictions['source'], predictions['target']], dtype = torch.long)

#   data["edge_label"] = torch.tensor(predictions['edge_type'], dtype = torch.long)

#   return data

# DATASET

In [ ]:
class FamilyDataset:
  def __init__(self, families, directed, num_relations) -> None:
    self.num_relations = num_relations
    self.directed = directed
    self.samples = []

    for family in families:
      for node in range(family['x'].size(0)):
        sample = make_masked_sample(family, node, directed)

        if sample is not None:
          self.samples.append(sample)

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    return self.samples[idx]

# MODEL

In [ ]:
class FamilyRelationGNN(torch.nn.Module):
  def __init__(self, in_dim, hidden_dim, num_relationships, dropout = 0.3):
    super().__init__()

    self.num_relations = num_relationships
    edge_dim = num_relationships

    self.input_proj = nn.Linear(in_dim, hidden_dim)

    # self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
    # self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
    self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
    self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
    self.dropout = nn.Dropout(dropout)

    self.classifier = nn.Sequential(
        nn.Linear(hidden_dim * 3, hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim, num_relationships)
    )

  def encode(self, x, edge_index, edge_attr):
    x = self.input_proj(x).relu()

    if edge_attr.size(0) > 0:
      edge_attr_oh = F.one_hot(edge_attr, num_classes = self.num_relations).float()
    else:
      edge_attr_oh = None
    x = self.dropout(x)
    x = self.conv1(x, edge_index, edge_attr = edge_attr_oh).relu()
    # x = self.dropout(x)
    # x = self.conv2(x, edge_index, edge_attr = edge_attr_oh).relu()

    return x

  def forward(self, data):
    x = self.encode(data.x, data.edge_index, data.edge_attr)

    src = data.edge_label_index[0]
    tgt = data.edge_label_index[1]

    x_src = self.input_proj(data.x[src]).relu()
    x_tgt = x[tgt]

    pair = torch.cat([x_src, x_tgt, x_src * x_tgt], dim = -1)

    return self.classifier(pair)

    # x = self.conv1(data.x, data.edge_index).relu()
    # x = self.conv2(x, data.edge_index)

    # src = data.edge_label_index[0]
    # dst = data.edge_label_index[1]

    # x_src = x[src]
    # x_dst = x[dst]

    # pair = torch.cat([x_src, x_dst, x_src * x_dst], dim = 1)

    # return self.classifier(pair)

In [ ]:
# class FamilyRelationGNN(torch.nn.Module):
#   def __init__(self, in_dim, hidden_dim, num_relationships, dropout = 0.3):
#     super().__init__()

#     self.num_relations = num_relationships
#     edge_dim = num_relationships

#     self.input_proj = nn.Linear(in_dim, hidden_dim)

#     # self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
#     # self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=False, edge_dim = edge_dim, dropout = dropout)
#     self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
#     self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=2, concat=False, edge_dim = edge_dim, dropout = dropout)
#     self.dropout = nn.Dropout(dropout)

#     self.classifier = nn.Sequential(
#         nn.Linear(hidden_dim * 3, hidden_dim),
#         nn.ReLU(),
#         nn.Dropout(dropout),
#         nn.Linear(hidden_dim, num_relationships)
#     )

#   def encode(self, x, edge_index, edge_attr):
#     # x = self.input_proj(x).relu()

#     if edge_attr.size(0) > 0:
#       edge_attr_oh = F.one_hot(edge_attr, num_classes = self.num_relations).float()
#     else:
#       edge_attr_oh = None
#     # x = self.dropout(x)
#     x = self.conv1(x, edge_index, edge_attr = edge_attr_oh).relu()
#     # x = self.dropout(x)
#     # x = self.conv2(x, edge_index, edge_attr = edge_attr_oh).relu()

#     return x

#   def forward(self, data):
#     x = self.encode(data.x, data.edge_index, data.edge_attr)

#     src = data.edge_label_index[0]
#     tgt = data.edge_label_index[1]

#     x_src = self.input_proj(data.x[src]).relu()
#     x_tgt = x[tgt]

#     pair = torch.cat([x_src, x_tgt, x_src * x_tgt], dim = -1)

#     return self.classifier(pair)

#     # x = self.conv1(data.x, data.edge_index).relu()
#     # x = self.conv2(x, data.edge_index)

#     # src = data.edge_label_index[0]
#     # dst = data.edge_label_index[1]

#     # x_src = x[src]
#     # x_dst = x[dst]

#     # pair = torch.cat([x_src, x_dst, x_src * x_dst], dim = 1)

#     # return self.classifier(pair)

# TRAINING

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer, device):
  model.train()
  total_loss, total_edges = 0, 0

  # counts_tensor = torch.tensor([counts[i] for i in range(num_relations)], dtype = torch.float)
  # weights = 1/counts_tensor
  # weights = (weights / weights.sum()).to(device)

  for data in tqdm(loader, desc = "Training", leave=False):
    data = data.to(device)
    optimizer.zero_grad()

    logits = model(data)

    # loss = F.cross_entropy(logits, data.edge_label, weight=weights)
    loss = F.cross_entropy(logits, data.edge_label)

    loss.backward()
    optimizer.step()

    n = data.edge_label.size(0)

    total_loss += loss.item() * n

    total_edges += n

  return total_loss / total_edges

In [ ]:
def evaluate(model, loader, device):
  model.eval()
  total_loss, total_edges = 0, 0

  all_preds, all_labels = [], []

  with torch.no_grad():
    for data in loader:
      data = data.to(device)

      logits = model(data)

      loss = F.cross_entropy(logits, data.edge_label)

      n = data.edge_label.size(0)
      total_loss += loss.item() * n
      total_edges += n

      all_preds.append(logits.argmax(dim = -1).cpu())
      all_labels.append(data.edge_label.cpu())

  all_preds = torch.cat(all_preds).numpy()
  all_labels = torch.cat(all_labels).numpy()

  avg_loss = total_loss / total_edges
  accuracy = (all_preds == all_labels).mean()
  macro_f1 = f1_score(all_labels, all_preds, average = "macro")
  report = classification_report(all_labels, all_preds, target_names = label_names, zero_division = 0)

  return avg_loss, accuracy, macro_f1, report

In [ ]:
from sklearn.utils import shuffle
def run_training(families, directed, num_relations, in_dim, hidden_dim = 64, epochs = 150, lr = 1e-3, weight_decay = 1e-4, batch_size = 32, val_split = 0.15, test_split = 0.15, patience = 20, label_names = None, device = None, save_path = "family_relations_model.pt"):
  if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  print(f"Using device: {device}")

  n = len(families)
  idx = torch.randperm(n).tolist()

  n_test = int(n * test_split)
  n_val = int(n * val_split)

  test_families = [families[i] for i in idx[:n_test]]
  val_families = [families[i] for i in idx[n_test:n_test + n_val]] if n_val > 0 else test_families
  train_families = [families[i] for i in idx[n_test + n_val:]]

  train_set = FamilyDataset(train_families, directed, num_relations)
  val_set = FamilyDataset(val_families, directed, num_relations)
  test_set = FamilyDataset(test_families, directed, num_relations)

  print(f"Families - train: {len(train_families)} | val: {len(val_families)} | test: {len(test_families)}")
  print(f"Samples - train: {len(train_set)}   | val: {len(val_set)}   | test: {len(test_set)}")

  # validate_dataset(train_set, num_relations, "train")
  # validate_dataset(val_set,   num_relations, "val")
  # validate_dataset(test_set,  num_relations, "test")

  train_loader = DataLoader(train_set, batch_size = batch_size, shuffle = True)
  val_loader = DataLoader(val_set, batch_size = batch_size)
  test_loader = DataLoader(test_set, batch_size = batch_size)

  model = FamilyRelationGNN(in_dim, hidden_dim, num_relations).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)

  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience = patience // 3, factor = 0.5)

  best_val_f1, best_state, no_improve = -1, None, 0

  for epoch in range(1, epochs + 1):
    print(f"Epoch {epoch}/{epochs}")

    train_loss = train_epoch(model, train_loader, optimizer, device)

    val_loss, val_acc, val_f1, _ = evaluate(model, val_loader, device)

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

    if val_f1 > best_val_f1:
      best_val_f1 = val_f1
      # best_state = copy.deepcopy(model.state_dict())
      best_state  = {k: v.clone() for k, v in model.state_dict().items()}
      no_improve = 0

    else:
      no_improve += 1

    if no_improve >= patience:
      print(f"Early stopping at epoch {epoch} - Best val F1: {best_val_f1:.4f}")
      break

  model.load_state_dict(best_state)
  test_loss, test_acc, test_f1, test_report = evaluate(model, test_loader, device)

  print(f"Test Loss: {test_loss:.4f} | Test F1: {test_f1:.4f} | Test Accuracy: {test_acc:.4f}")
  print(test_report)

  torch.save(model.state_dict(), save_path)
  print(f"Model saved to {save_path}")

  return model


In [ ]:
def find_asymmetric_pairs_from_files():
    asymmetric = set()
    family_ids = sorted([p.name for p in familyIDsDir.iterdir()])

    for id in tqdm(family_ids, desc="Scanning families"):
        mid_csv = familyIDsDir / id / "mid.csv"
        if not mid_csv.exists():
            continue

        try:
            df = pd.read_csv(mid_csv)
            mids = df["MID"].astype(int).tolist()

            # Build lookup: mid -> row
            row_map = {int(row["MID"]): row for _, row in df.iterrows()}

            for mid_a in mids:
                for mid_b in mids:
                    if mid_a == mid_b:
                        continue
                    if mid_a not in row_map or mid_b not in row_map:
                        continue

                    val_ab = row_map[mid_a].get(str(mid_b))
                    val_ba = row_map[mid_b].get(str(mid_a))

                    if not valid_relationship(val_ab) or not valid_relationship(val_ba):
                        continue

                    rel_ab = int(val_ab)
                    rel_ba = int(val_ba)

                    if rel_ab != rel_ba:
                        pair = tuple(sorted([rel_ab, rel_ba]))
                        asymmetric.add(pair)

        except Exception as e:
            print(f"\nFailed on {id}: {e}")

    return asymmetric

asymmetric_pairs = find_asymmetric_pairs_from_files()
print("Asymmetric relationship pairs (directed):")
for a, b in sorted(asymmetric_pairs):
    print(f"  {a} <-> {b}")


Scanning families: 100%|██████████| 981/981 [00:00<00:00, 1431.60it/s]

Asymmetric relationship pairs (directed):
  1 <-> 4
  3 <-> 6


In [ ]:
directed = False
families = load_all_families(directed)
num_relations = max(r for f in families for _, _, r in f['edges']) + 1

Loading Families: 100%|██████████| 980/980 [00:02<00:00, 469.77it/s]


In [ ]:
print(f"num_relations: {num_relations}")

num_relations: 4


In [ ]:
from collections import Counter

all_labels = [r for f in families for _, _, r in f['edges']]
counts = Counter(all_labels)
for rel, count in sorted(counts.items()):
    print(f"  rel {rel}: {count}")


  rel 0: 4770
  rel 1: 2295
  rel 2: 9518
  rel 3: 2232


In [ ]:
directed = True
families = load_all_families(directed)
num_relations = max(r for f in families for _, _, r in f['edges']) + 1

from collections import Counter

all_labels = [r for f in families for _, _, r in f['edges']]
counts = Counter(all_labels)
for rel, count in sorted(counts.items()):
    print(f"  rel {rel}: {count}")


Loading Families:   0%|          | 0/980 [00:00<?, ?it/s]

Loading Families: 100%|██████████| 980/980 [00:01<00:00, 555.10it/s]


  rel 0: 4760
  rel 1: 4770
  rel 2: 1147
  rel 3: 4758
  rel 4: 2232
  rel 5: 1148


In [ ]:

directed = True
num_relations = 6
# label_names = ["2 - sibling", "3 / 6", "1 / 4 - parent / child", "5 - spouse", "7 / 8"]
# num_relations = 8
# label_names = ['1', '2', '3', '4', '5', '6', '7', '8']
label_names = ['1', '2', '3', '4', '5', '6']

families = load_all_families(directed)
in_dim   = families[0]['x'].size(1)

model = run_training(
  families = families,
  directed = directed,
  num_relations = num_relations,
  in_dim = in_dim,
  label_names = label_names,
  # hidden_dim = 96,
  hidden_dim = 256,
  batch_size = 16,
  # val_split = 0,
  # test_split=0.1,
  # save_path = "family_relations_model_WEIGHTED_no_78.pt"
  save_path = "family_relations_model_no_78_NO_DIM_RED.pt"
)


Loading Families: 100%|██████████| 980/980 [00:00<00:00, 1150.55it/s]


Using device: cpu
Families - train: 686 | val: 147 | test: 147
Samples - train: 3507   | val: 757   | test: 745
Epoch 1/150


Train Loss: 1.3661 | Val Loss: 0.9836 | Val F1: 0.4944 | Val Acc: 0.6347
Epoch 2/150


Train Loss: 0.9014 | Val Loss: 0.8313 | Val F1: 0.6529 | Val Acc: 0.7006
Epoch 3/150


Train Loss: 0.7618 | Val Loss: 0.7914 | Val F1: 0.6577 | Val Acc: 0.7204
Epoch 4/150


...

Train Loss: 0.1777 | Val Loss: 1.1156 | Val F1: 0.6311 | Val Acc: 0.6953
Epoch 21/150


Train Loss: 0.1717 | Val Loss: 1.1208 | Val F1: 0.6338 | Val Acc: 0.6953
Epoch 22/150


Train Loss: 0.1657 | Val Loss: 1.1479 | Val F1: 0.6356 | Val Acc: 0.6932
Epoch 23/150


Train Loss: 0.1621 | Val Loss: 1.1834 | Val F1: 0.6248 | Val Acc: 0.6869
Epoch 24/150


Train Loss: 0.1544 | Val Loss: 1.1356 | Val F1: 0.6387 | Val Acc: 0.6943
Early stopping at epoch 24 - Best val F1: 0.6712
Test Loss: 0.7882 | Test F1: 0.6547 | Test Accuracy: 0.7058
              precision    recall  f1-score   support

           1       0.80      0.79      0.80       730
           2       0.82      0.62      0.70       688
           3       0.65      0.52      0.58       174
           4       0.70      0.83      0.76       730
           5       0.48      0.64      0.55       322
           6       0.56      0.52      0.54       174

    accuracy                           0.71      2818
   macro avg       0.67      0.65      0.65      2818
weighted avg       0.72      0.71      0.71      2818

Model saved to family_relations_model_no_78_NO_DIM_RED.pt


In [ ]:

directed = False
num_relations = 5
label_names = ["2 - sibling", "3 / 6", "1 / 4 - parent / child", "5 - spouse", "7 / 8"]
# num_relations = 8
# label_names = ['0', '1', '2', '3', '4', '5', '6', '7']

families = load_all_families(directed)
in_dim   = families[0]['x'].size(1)

model = run_training(
  families = families,
  directed = directed,
  num_relations = num_relations,
  in_dim = in_dim,
  label_names = label_names,
)


Loading Families: 100%|██████████| 980/980 [00:02<00:00, 468.52it/s]


Using device: cuda
Families - train: 686 | val: 147 | test: 147
Samples - train: 3487   | val: 760   | test: 762
Epoch 1/150


Train Loss: 1.3200 | Val Loss: 1.2461 | Val F1: 0.1327
Epoch 2/150


Train Loss: 1.2342 | Val Loss: 1.2248 | Val F1: 0.1327
Epoch 3/150


Train Loss: 1.2006 | Val Loss: 1.1883 | Val F1: 0.1327
Epoch 4/150


...

Train Loss: 0.6058 | Val Loss: 0.9155 | Val F1: 0.5069
Epoch 85/150


Train Loss: 0.6067 | Val Loss: 0.9155 | Val F1: 0.5073
Epoch 86/150


Train Loss: 0.6062 | Val Loss: 0.9153 | Val F1: 0.5064
Epoch 87/150


Train Loss: 0.6071 | Val Loss: 0.9151 | Val F1: 0.5063
Epoch 88/150


Train Loss: 0.5975 | Val Loss: 0.9155 | Val F1: 0.5062
Early stopping at epoch 88 - Best val F1: 0.5104
Test Loss: 1.0825 | Test F1: 0.4548
                        precision    recall  f1-score   support

           2 - sibling       0.62      0.66      0.64       730
                 3 / 6       0.47      0.45      0.46       394
1 / 4 - parent / child       0.66      0.70      0.68      1456
            5 - spouse       0.50      0.48      0.49       358
                 7 / 8       0.00      0.00      0.00       100

              accuracy                           0.61      3038
             macro avg       0.45      0.46      0.45      3038
          weighted avg       0.59      0.61      0.60      3038

Model saved to family_relations_model.pt


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
def validate_dataset(dataset, num_relations, name="dataset"):
    errors = []
    for i, data in enumerate(dataset):
        if data.edge_attr.numel() > 0:
            if data.edge_attr.min() < 0 or data.edge_attr.max() >= num_relations:
                errors.append(f"Sample {i}: edge_attr out of range — min={data.edge_attr.min()}, max={data.edge_attr.max()}")
        if data.edge_label.min() < 0 or data.edge_label.max() >= num_relations:
            errors.append(f"Sample {i}: edge_label out of range — min={data.edge_label.min()}, max={data.edge_label.max()}")

    if errors:
        print(f"\n{name} validation FAILED:")
        for e in errors:
            print(f"  {e}")
    else:
        print(f"{name}: all samples valid")

# validate_dataset(train_ds, num_relations, "train")
# validate_dataset(val_ds,   num_relations, "val")
# validate_dataset(test_ds,  num_relations, "test")

In [ ]:
class HeteroRelationshipGNN(nn.Module):
  def __init__(self, in_dim, hidden_dim, graph_labels, num_classes, dropout = 0.3, directed = True):
    super().__init__()

    self.graph_labels = graph_labels
    self.droupout = dropout

    self.input_proj = nn.linear(in_dim, hidden_dim)

    edge_types

In [194]:
import torch.nn as nn
from torch_geometric.nn import SAGEConv, global_add_pool, GINConv
from torch_geometric.utils import negative_sampling, add_self_loops

def gin_mlp(in_d, out_d):
            return nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.LeakyReLU(0.1),
                nn.Linear(out_d, out_d)
            )
            
class GatingSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GatingSAGE, self).__init__()

        self.conv1 = GINConv(gin_mlp(in_channels, hidden_channels))
        self.conv2 = GINConv(gin_mlp(hidden_channels, hidden_channels))
        
        # We concatenate h1 and h2, so node embedding size is 2 * hidden
        self.z_dim = hidden_channels * 2

        self.link_predictor = nn.Sequential(
            nn.Linear(self.z_dim * 3, hidden_channels * 2),
            nn.BatchNorm1d(hidden_channels * 2),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.LeakyReLU(0.1),
            nn.Linear(hidden_channels, 1)
        )

    def encode(self, x, edge_index):
        h1 = self.conv1(x, edge_index) # Removed .relu()
        h2 = self.conv2(h1, edge_index)
        return torch.cat([h1, h2], dim=-1)

    def decode(self, z, edge_label_index):
        src, dst = z[edge_label_index[0]], z[edge_label_index[1]]
        
        s_sum = src + dst
        diff = torch.abs(src - dst)
        mult = src * dst
        
        return self.link_predictor(torch.cat([s_sum, diff, mult], dim=-1))

# Calculate weights based on your average family density
# In FIW, negative pairs usually outnumber positive pairs in a full batch
pos_weight = torch.tensor([5]) 
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [195]:
import torch
from torch_geometric.loader import DataLoader
from tqdm import tqdm
from sklearn.metrics import recall_score, precision_score, f1_score
from sklearn.model_selection import train_test_split

train_fams, test_fams = train_test_split(families, test_size=0.2, random_state=42)

def create_pyg_data(family_dict):
    x = family_dict['x']
    # Convert edges to directed (source, target) for PyG
    edge_index = torch.tensor([(e[0], e[1]) for e in family_dict['edges']], dtype=torch.long).t()
    return Data(x=x, edge_index=edge_index, family_id=family_dict['family_id'])

train_data_list = [create_pyg_data(f) for f in train_fams]
test_data_list = [create_pyg_data(f) for f in test_fams]

In [187]:
import torch
from torch_geometric.loader import DataLoader
from tqdm import tqdm
from sklearn.metrics import recall_score, precision_score, f1_score
from torch_geometric.utils import negative_sampling, is_undirected, to_undirected

# 4 times negative sampling 
def train_step(model, data, optimizer, criterion, device, neg_sampling=4):
    model.train()
    optimizer.zero_grad()

    # 1. Split edges: Use 70% for message passing, 30% for supervision
    num_edges = data.edge_index.size(1)
    
    # Ensure random permutation happens on the correct device
    perm = torch.randperm(num_edges, device=device) 
    
    split_idx = int(num_edges * 0.7)
    msg_passing_edges = data.edge_index[:, perm[:split_idx]]
    pos_target_edges = data.edge_index[:, perm[split_idx:]]

    # 2. Encode using ONLY the message passing edges
    z = model.encode(data.x, msg_passing_edges)

    # 3. Sample negatives for the supervision set
    from torch_geometric.utils import negative_sampling
    # Passing data.edge_index ensures we don't accidentally sample ANY true edges as negatives
    neg_target_edges = negative_sampling(
        data.edge_index, 
        num_nodes=data.num_nodes, 
        num_neg_samples=pos_target_edges.size(1) * neg_sampling
    )

    # 4. Decode and Loss
    edge_label_index = torch.cat([pos_target_edges, neg_target_edges], dim=-1)
    labels = torch.cat([
        torch.ones(pos_target_edges.size(1)), 
        torch.zeros(neg_target_edges.size(1))
    ], dim=0).to(device)

    logits = model.decode(z, edge_label_index).view(-1)
    
    # Use the criterion passed from the main loop
    loss = criterion(logits, labels) 
    
    loss.backward()
    optimizer.step()
    
    return loss.item()

import torch
from sklearn.metrics import recall_score, precision_score

@torch.no_grad()
def validate_gating_model(model, test_loader, threshold=0.3):
    model.eval()
    total_recall = 0
    total_precision = 0
    all_preds = []
    all_labels = []

    for data in test_loader:
        # 1. Encode the test family graph
        z = model.encode(data.x, data.edge_index)
        
        # 2. Prepare Ground Truth (Positives) and Negative Samples
        from torch_geometric.utils import negative_sampling
        neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes)
        
        edge_label_index = torch.cat([data.edge_index, neg_edge_index], dim=-1)
        labels = torch.cat([
            torch.ones(data.edge_index.size(1)), 
            torch.zeros(neg_edge_index.size(1))
        ], dim=0)

        # 3. Inference
        probs = model.decode(z, edge_label_index).view(-1)
        preds = (probs > threshold).float()

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    # Flatten results for metric calculation
    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()

    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    
    # print(f"Validation Results (Threshold {threshold}):")
    # print(f"--- Recall (Sensitivity): {recall:.4f}")
    # print(f"--- Precision: {precision:.4f}")
    return recall, precision

@torch.no_grad()
def evaluate_gating(model, loader, criterion, device, threshold=0.3):
    model.eval()
    all_labels, all_preds, losses = [], [], []
    
    for data in loader:
        data = data.to(device)
        
        # 1. To prevent bidirectional leakage, only keep upper triangle edges 
        # (row < col) so we only split unique pairs, not reverse edges
        row, col = data.edge_index
        mask = row < col
        unique_edges = data.edge_index[:, mask]
        
        num_unique_edges = unique_edges.size(1)
        perm = torch.randperm(num_unique_edges, device=device)
        split_idx = int(num_unique_edges * 0.7)
        
        # Split the unique edges
        msg_edges_half = unique_edges[:, perm[:split_idx]]
        pos_target_edges_half = unique_edges[:, perm[split_idx:]]
        
        # 2. Re-create undirected edges ONLY for the message passing set
        # This guarantees the target edges (and their reverse counterparts) are 100% hidden
        msg_passing_edges = to_undirected(msg_edges_half, num_nodes=data.num_nodes)
        
        # We can evaluate on the half-edges, or make target edges undirected too. 
        # Let's keep the target edges directed to evaluate each unique pair exactly once.
        pos_target_edges = pos_target_edges_half
        
        # 3. Encode using strictly hidden message passing edges
        z = model.encode(data.x, msg_passing_edges)
        
        # 4. Sample negatives using the FULL original edge index to avoid sampling true edges
        neg_target_edges = negative_sampling(
            data.edge_index, 
            num_nodes=data.num_nodes,
            num_neg_samples=pos_target_edges.size(1)
        )
        
        edge_label_index = torch.cat([pos_target_edges, neg_target_edges], dim=-1)
        labels = torch.cat([
            torch.ones(pos_target_edges.size(1)), 
            torch.zeros(neg_target_edges.size(1))
        ], dim=0).to(device)
        
        logits = model.decode(z, edge_label_index).view(-1)
        loss = criterion(logits, labels)
        
        preds = (torch.sigmoid(logits) > threshold).float()
        
        all_labels.append(labels.cpu())
        all_preds.append(preds.cpu())
        losses.append(loss.item())

    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()
    f1 = f1_score(y_true, y_pred)
    
    return recall_score(y_true, y_pred), precision_score(y_true, y_pred), f1, sum(losses)/len(losses)

def run_gating_training(families, in_dim, hidden_dim=64, epochs=150, lr=1e-3, 
                        weight_decay=1e-4, batch_size=32, val_split=0.15, 
                        test_split=0.15, patience=20, device=None, 
                        save_path="gating_model.pt"):
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Family-wise Split (using your existing index logic)
    n = len(families)
    idx = torch.randperm(n).tolist()
    n_test, n_val = int(n * test_split), int(n * val_split)

    # Convert your list of dicts to PyG Data primitives
    test_data = [create_pyg_data(families[i]) for i in idx[:n_test]]
    val_data = [create_pyg_data(families[i]) for i in idx[n_test:n_test + n_val]]
    train_data = [create_pyg_data(families[i]) for i in idx[n_test + n_val:]]

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    test_loader = DataLoader(test_data, batch_size=batch_size)

    # 2. Model & Optimizer
    model = GatingSAGE(in_dim, hidden_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Weighting the positive class heavily (e.g., 5.0) to force the gate to be inclusive
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=patience//3, factor=0.5)

    best_val_f1, best_state, no_improve = -1, None, 0

    # 3. Training Loop
    total_steps = epochs * len(train_loader)

# Create one master progress bar
    with tqdm(total=total_steps, desc="Training", unit="batch") as pbar:
        for epoch in range(1, epochs + 1):
            model.train()
            total_loss = 0
            
            # tqdm for training batches
            # pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", unit="batch")
            for data in train_loader:
                data = data.to(device)
                # Execute the clean train_step
                loss_val = train_step(model, data, optimizer, criterion, device)
                
                pbar.update(1)
                total_loss += loss_val
                pbar.set_postfix(loss=loss_val, refresh=False)

            # 4. Validation Step (using the validation primitive logic)
            val_recall, val_precision, val_f1, val_loss = evaluate_gating(model, val_loader, criterion, device)
            scheduler.step(val_loss)

            pbar.set_postfix({ "Loss": val_loss, "F1": val_f1, "Recall": val_recall, "Precision": val_precision })
            # Now, only save if F1 improves, or if Recall is high AND Precision is improving
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                torch.save(best_state, save_path)
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                print(f"Early stopping. Best F1: {best_val_f1:.4f}")
                break
    

    # 5. Final Test
    model.load_state_dict(best_state)
    test_recall, test_prec, test_f1, test_loss = evaluate_gating(model, test_loader, criterion, device)
    print(f"\nFINAL TEST RESULTS:\nRecall: {test_recall:.4f} | Precision: {test_prec:.4f} | F1: {test_f1:.4f}")
    return model


In [196]:
model = run_gating_training(
    families = families,
    in_dim = in_dim,
    hidden_dim = 256,
    epochs = 200,
    lr = 1e-3,
    weight_decay = 1e-4,
    batch_size = 16,
    val_split = 0.10,
    test_split = 0.10,
    patience = 40,
    device = None,
    save_path = "gating_model.pt"
)

Training:  56%|█████▌    | 5488/9800 [00:45<00:35, 121.16batch/s, Loss=1.18, F1=0.949, Recall=0.944, Precision=0.953] 

Early stopping. Best F1: 0.9704

FINAL TEST RESULTS:
Recall: 0.9251 | Precision: 0.9589 | F1: 0.9417


In [197]:
from torch_geometric.data import Data, Batch
%reload_ext autoreload
# Reload fp
import fiw_processor as fp

image_emb = fp.get_face_embedding("ben-afflect.jpg", app=fp.get_face_app())
new_face_vector = torch.tensor(image_emb).unsqueeze(0)  # Shape: [1, in_dim]
print(new_face_vector.shape)



/Users/fedeshih/Repositories/kinship-trees-gnn/.venv/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CoreMLExecutionProvider': {}}
find model: /Users/fedeshih/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CoreMLExecutionProvider': {}}
find model: /Users/fedeshih/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CoreMLExecutionProvider': {}}
find model: /Users/fedeshih/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CoreMLExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CoreMLExecutionProvider': {}}
find model: /Users/fedeshih/.insightface/models/buffalo_l/genderage.onnx 

/Users/fedeshih/Repositories/kinship-trees-gnn/.venv/lib/python3.13/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


In [200]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_scatter import scatter_max

@torch.no_grad()
def identify_family(new_face_vector, families_data, gating_model, device, batch_size=64, threshold=0.3):
    """
    new_face_vector: [1, in_dim] - The ArcFace vector of the probe
    families_data: List of PyG Data objects created from families
    """
    gating_model.eval()
    new_face_vector = new_face_vector.to(device)
    
    # Use a DataLoader to group family graphs into batches
    loader = DataLoader(families_data, batch_size=batch_size, shuffle=False)
    
    all_rankings = []

    # 1. Encode the new face ONCE (Optimization)
    # We give it a self-loop so the GNN processes its magnitude correctly
    edge_index_new = torch.zeros((2, 1), dtype=torch.long).to(device)
    z_new = gating_model.encode(new_face_vector, edge_index_new)

    for batch in loader:
        batch = batch.to(device)
        
        # 2. Encode all families in the current batch
        z_families = gating_model.encode(batch.x, batch.edge_index)
        
        # 3. Combine the embeddings into a single tensor for the decode function
        # We append z_new to the end. Its index will be `num_family_nodes`
        num_family_nodes = z_families.size(0)
        z_combined = torch.cat([z_families, z_new], dim=0)
        
        # 4. Create an edge_label_index connecting the probe to every family node
        # Row 0: The probe's index. Row 1: The target family nodes' indices (0 to N-1).
        probe_indices = torch.full((num_family_nodes,), num_family_nodes, dtype=torch.long, device=device)
        target_indices = torch.arange(num_family_nodes, dtype=torch.long, device=device)
        
        edge_label_index = torch.stack([probe_indices, target_indices], dim=0)
        
        # 5. Predict links using your model's native decode function!
        logits = gating_model.decode(z_combined, edge_label_index).view(-1)
        probs = torch.sigmoid(logits)
        
        # 6. Aggregate results by family using scatter_max
        family_max_probs, _ = scatter_max(probs, batch.batch, dim=0)
        
        # 7. Store results
        unique_fids = batch.family_id 
        
        for i, fid in enumerate(unique_fids):
            score = family_max_probs[i].item()
            if score > threshold:
                all_rankings.append({'family_id': fid, 'confidence': score})

    # Final sort across all batches
    return sorted(all_rankings, key=lambda x: x['confidence'], reverse=True)

families_data = load_all_families(directed)

identify_family(new_face_vector, families_data, model, device, threshold=0.4)

Loading Families: 100%|██████████| 980/980 [00:02<00:00, 447.44it/s]


[{'family_id': 'F0992', 'confidence': 1.0},
 {'family_id': 'F0016', 'confidence': 0.9160900115966797},
 {'family_id': 'F0128', 'confidence': 0.8143770694732666},
 {'family_id': 'F0165', 'confidence': 0.8139735460281372},
 {'family_id': 'F0192', 'confidence': 0.8049417734146118},
 {'family_id': 'F0090', 'confidence': 0.7358615398406982},
 {'family_id': 'F0601', 'confidence': 0.7291925549507141},
 {'family_id': 'F0106', 'confidence': 0.6632828116416931},
 {'family_id': 'F0074', 'confidence': 0.6581674814224243},
 {'family_id': 'F0785', 'confidence': 0.6525489091873169},
 {'family_id': 'F0082', 'confidence': 0.6379169821739197},
 {'family_id': 'F0023', 'confidence': 0.5504931807518005},
 {'family_id': 'F0037', 'confidence': 0.5034723877906799},
 {'family_id': 'F0103', 'confidence': 0.49513915181159973}]